# Test Gemma 3 Adapter with Sentiment Steering

This notebook tests the improved Gemma 3 adapter with proper vector normalization.

**Goals:**
- Train a simple happy-sad sentiment vector
- Verify that coefficients in the range 0.5-5.0 work (not 300-500!)
- Compare normalized vs. non-normalized vectors
- Test both standard and adaptive scaling

## 1. Setup

In [1]:
import sys
import json
import torch
import numpy as np
from pathlib import Path

# Add paths
sys.path.insert(0, '../repeng')
sys.path.insert(0, '.')

# Import our adapter
from gemma3_adapter import (
    Gemma3ControlVector,
    Gemma3ControlModel,
    create_gemma3_model,
    make_dataset_with_truncation,
    print_vector_analysis
)

from repeng import DatasetEntry

print("✓ Imports successful!")

/Users/ivanculo/Desktop/Projects/Cogni_map/brije/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Imports successful!


## 2. Load Model

In [2]:
# Create Gemma 3 model with proper configuration
model, tokenizer = create_gemma3_model(
    model_name="google/gemma-3-4b-it",
    layer_range=(-4, -20),  # Layers -4 to -20 inclusive
    use_bfloat16=True
)

print(f"\nModel device: {model.device}")
print(f"Model dtype: {model.model.dtype}")
print(f"Controlling {len(model.layer_ids)} layers: {model.layer_ids}")

Loading google/gemma-3-4b-it...
Using dtype: torch.bfloat16


Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.83s/it]


Fixing missing num_hidden_layers attribute (setting to 34)
Model loaded on device: mps:0
Total layers: 34
Wrapping layers: [-4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20]

Model device: mps:0
Model dtype: torch.bfloat16
Controlling 17 layers: [30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14]


## 3. Load Training Data

In [3]:
# Load output suffixes from your ToM data
suffixes_path = Path("/Users/ivanculo/Desktop/Projects/Cogni_map/brije/ToM/repeng/notebooks/data/all_truncated_outputs.json")

if suffixes_path.exists():
    with open(suffixes_path) as f:
        output_suffixes = json.load(f)
    print(f"Loaded {len(output_suffixes)} output suffixes")
else:
    # Fallback: create simple suffixes
    output_suffixes = [
        "That's interesting.",
        "I can see that.",
        "Let me think about it.",
        "That makes sense.",
        "I understand."
    ] * 20  # Repeat to get ~100 suffixes
    print(f"Using {len(output_suffixes)} fallback suffixes")

print(f"Examples: {output_suffixes[:3]}")

Loaded 582 output suffixes
Examples: ['', 'That game', 'I can see']


## 4. Create Training Dataset

Simple happy-sad sentiment pairs.

In [4]:
# Create sentiment dataset with truncation (following repeng best practices)
sentiment_dataset = make_dataset_with_truncation(
    template="Act as if you're extremely {persona}.",
    pos_personas=["happy", "excited"],
    neg_personas=["sad", "depressed"],
    suffixes=output_suffixes[:400],  # Use subset for faster training
    truncate_suffixes=False,
    max_truncations=5
)

print(f"Created {len(sentiment_dataset)} training pairs")
print(f"\nExample pair:")
print(f"Positive: {sentiment_dataset[0].positive[:100]}...")
print(f"Negative: {sentiment_dataset[0].negative[:100]}...")

Created 800 training pairs

Example pair:
Positive: Act as if you're extremely happy. ...
Negative: Act as if you're extremely sad. ...


## 5. Train Vectors

We'll train three versions:
1. Standard (normalized, with activation measurement)
2. Without normalization (to compare)
3. Without activation measurement

In [5]:
print("="*80)
print("Training NORMALIZED vector with activation measurement")
print("="*80)

model.reset()
normalized_vector = Gemma3ControlVector.train(
    model,
    tokenizer,
    sentiment_dataset,
    method='pca_center',
    measure_activations=True,
    normalize_vectors=True,
    batch_size=16
)

print("\nVector trained!")
print_vector_analysis(normalized_vector)

Training NORMALIZED vector with activation measurement


100%|██████████| 33/33 [00:01<00:00, 31.95it/s]


Measuring baseline activation magnitudes...


100%|██████████| 5/5 [00:01<00:00,  4.37it/s]

Activation norms: {30: 63496.2109375, 29: 66308.0234375, 28: 63085.41015625, 27: 57006.49609375, 26: 51969.4609375, 25: 47499.76171875, 24: 44546.25, 23: 43598.3203125, 22: 42874.66796875, 21: 39133.3125, 20: 36786.625, 19: 34101.75, 18: 32536.703125, 17: 28721.4375, 16: 25635.62890625, 15: 23417.5, 14: 23087.482421875}

Vector trained!

VECTOR ANALYSIS
Number of layers: 33
Layer IDs: [33, 32, 31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]

Vector Norms (should be ~1.0 if normalized):
  Layer   1: 1.000000
  Layer   2: 1.000000
  Layer   3: 1.000000
  Layer   4: 1.000000
  Layer   5: 1.000000
  Layer   6: 1.000000
  Layer   7: 1.000000
  Layer   8: 1.000000
  Layer   9: 1.000000
  Layer  10: 1.000000
  Layer  11: 1.000000
  Layer  12: 1.000000
  Layer  13: 1.000000
  Layer  14: 1.000000
  Layer  15: 1.000000
  Layer  16: 1.000000
  Layer  17: 1.000000
  Layer  18: 1.000000
  Layer  19: 1.000000
  Layer  20: 1.000000
  

In [6]:
print("="*80)
print("Training NON-NORMALIZED vector (for comparison)")
print("="*80)

model.reset()
unnormalized_vector = Gemma3ControlVector.train(
    model,
    tokenizer,
    sentiment_dataset,
    method='pca_center',
    measure_activations=False,
    normalize_vectors=False,
    batch_size=16
)

print("\nVector trained!")
print_vector_analysis(unnormalized_vector)

Training NON-NORMALIZED vector (for comparison)


100%|██████████| 33/33 [00:00<00:00, 34.36it/s]


Vector trained!

VECTOR ANALYSIS
Number of layers: 33
Layer IDs: [33, 32, 31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]

Vector Norms (should be ~1.0 if normalized):
  Layer   1: 1.000000
  Layer   2: 1.000000
  Layer   3: 1.000000
  Layer   4: 1.000000
  Layer   5: 1.000000
  Layer   6: 1.000000
  Layer   7: 1.000000
  Layer   8: 1.000000
  Layer   9: 1.000000
  Layer  10: 1.000000
  Layer  11: 1.000000
  Layer  12: 1.000000
  Layer  13: 1.000000
  Layer  14: 1.000000
  Layer  15: 1.000000
  Layer  16: 1.000000
  Layer  17: 1.000000
  Layer  18: 1.000000
  Layer  19: 1.000000
  Layer  20: 1.000000
  Layer  21: 1.000000
  Layer  22: 1.000000
  Layer  23: 1.000000
  Layer  24: 1.000000
  Layer  25: 1.000000
  Layer  26: 1.000000
  Layer  27: 1.000000
  Layer  28: 1.000000
  Layer  29: 1.000000
  Layer  30: 1.000000
  Layer  31: 1.000000
  Layer  32: 1.000000
  Layer  33: 1.000000



## 6. Test Generation

Test both vectors with reasonable coefficients (0.5-5.0 range).

In [7]:
def generate_text(prompt, model, tokenizer, max_new_tokens=80):
    """Generate text using chat template format."""
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )
    input_ids = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    output = model.generate(
        input_ids.input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.8,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.1
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

test_prompt = "How was your day?"

### 6.1 Test Normalized Vector

In [8]:
print("="*80)
print("TESTING NORMALIZED VECTOR")
print("="*80)

# Baseline
print("\n[BASELINE - No Steering]")
print("-"*80)
model.reset()
baseline = generate_text(test_prompt, model, tokenizer)
print(baseline)

# Test different coefficients
test_coefficients = [2.0, 3.0, 10.0]

for coeff in test_coefficients:
    print(f"\n{'='*80}")
    print(f"[POSITIVE STEERING - Coefficient: {coeff}]")
    print(f"Expected: Happy, cheerful response")
    print("-"*80)
    model.set_control(normalized_vector, coeff=coeff)
    positive = generate_text(test_prompt, model, tokenizer)
    print(positive)

for coeff in test_coefficients:
    print(f"\n{'='*80}")
    print(f"[NEGATIVE STEERING - Coefficient: -{coeff}]")
    print(f"Expected: Sad, melancholy response")
    print("-"*80)
    model.set_control(normalized_vector, coeff=-coeff)
    negative = generate_text(test_prompt, model, tokenizer)
    print(negative)

model.reset()

TESTING NORMALIZED VECTOR

[BASELINE - No Steering]
--------------------------------------------------------------------------------
user
How was your day?
model
As an AI, I don't really *have* days in the same way humans do! But my systems have been running smoothly and I’ve been busy helping lots of people with all sorts of things – answering questions, writing stories, translating languages, summarizing text, and even just chatting. 

It’s been a productive time for me learning and assisting. 😊 

Thanks for

[POSITIVE STEERING - Coefficient: 2.0]
Expected: Happy, cheerful response
--------------------------------------------------------------------------------
user
How was your day?
model
As an AI, I don't really *have* days in the same way humans do! But my systems have been running smoothly and I’ve been busy helping lots of people with all sorts of questions and tasks. 😊 

I've answered questions about everything from recipes to historical events, written stories, translated lang

### 6.2 Test Unnormalized Vector (with appropriate coefficients)

In [9]:
print("="*80)
print("TESTING UNNORMALIZED VECTOR")
print("This should require MUCH larger coefficients (like 300-500)")
print("="*80)

# Calculate approximate scaling factor
# Compare norms of first layer
first_layer = list(normalized_vector.directions.keys())[0]
norm_ratio = (
    np.linalg.norm(unnormalized_vector.directions[first_layer]) /
    np.linalg.norm(normalized_vector.directions[first_layer])
)
print(f"\nNorm ratio (unnormalized/normalized): {norm_ratio:.6f}")
print(f"Suggested coefficient scaling: ~{1/norm_ratio:.2f}x")

# Test with scaled coefficients
unnorm_coeffs = [1.0, 2.0, 3.0]  # These likely won't work
scaled_coeffs = [300, 400, 500]  # These might be needed

print("\n--- Testing with small coefficients (likely won't work) ---")
for coeff in unnorm_coeffs:  # Just test one
    print(f"\n[POSITIVE - Coefficient: {coeff}]")
    model.set_control(unnormalized_vector, coeff=coeff)
    output = generate_text(test_prompt, model, tokenizer, max_new_tokens=50)
    print(output[:200] + "...")

print("\n--- Testing with large coefficients (the old way) ---")
for coeff in scaled_coeffs[:1]:  # Just test one
    print(f"\n[POSITIVE - Coefficient: {coeff}]")
    model.set_control(unnormalized_vector, coeff=coeff)
    output = generate_text(test_prompt, model, tokenizer, max_new_tokens=50)
    print(output[:200] + "...")

model.reset()

TESTING UNNORMALIZED VECTOR
This should require MUCH larger coefficients (like 300-500)

Norm ratio (unnormalized/normalized): 1.000000
Suggested coefficient scaling: ~1.00x

--- Testing with small coefficients (likely won't work) ---

[POSITIVE - Coefficient: 1.0]
user
How was your day?
model
As an AI, I don't really *have* days in the same way humans do! But my systems have been running smoothly and I’ve been busy helping people with all sorts of things – answ...

[POSITIVE - Coefficient: 2.0]
user
How was your day?
model
As an AI, I don't really *have* days in the same way humans do! But my activity level has been pretty busy today. I’ve answered lots of questions on a huge range of topics...

[POSITIVE - Coefficient: 3.0]
user
How was your day?
model
As an AI, I don't really *have* days in the same way humans do! But my systems have been running smoothly and I’ve been busy assisting users with all sorts of requests – a...

--- Testing with large coefficients (the old way) ---

[POS

### 6.3 Test Adaptive Scaling

In [10]:
print("="*80)
print("TESTING ADAPTIVE SCALING (per-layer coefficients)")
print("="*80)

if normalized_vector.activation_norms:
    # Convert model to Gemma3ControlModel if needed
    if not isinstance(model, Gemma3ControlModel):
        # Wrap the existing ControlModel
        base = model.model
        model = Gemma3ControlModel(base, model.layer_ids)
    
    print("\n[ADAPTIVE SCALING - Base coefficient: 2.0]")
    print("(Automatically adjusts per layer based on activation magnitude)")
    print("-"*80)
    
    model.set_control_per_layer(
        normalized_vector,
        base_coeff=2.0,
        use_adaptive_scaling=True
    )
    
    # Show the actual coefficients used
    print("\nPer-layer coefficients:")
    for layer_id in sorted(model.layer_ids):
        coeff = normalized_vector.get_scaled_coefficient(layer_id, base_coeff=2.0)
        print(f"  Layer {layer_id:3d}: {coeff:.4f}")
    
    print("\n" + "-"*80)
    adaptive = generate_text(test_prompt, model, tokenizer)
    print(adaptive)
    
    model.reset()
else:
    print("No activation norms available. Retrain with measure_activations=True")

TESTING ADAPTIVE SCALING (per-layer coefficients)

[ADAPTIVE SCALING - Base coefficient: 2.0]
(Automatically adjusts per layer based on activation magnitude)
--------------------------------------------------------------------------------

Per-layer coefficients:
  Layer  14: 0.0086
  Layer  15: 0.0084
  Layer  16: 0.0078
  Layer  17: 0.0070
  Layer  18: 0.0062
  Layer  19: 0.0059
  Layer  20: 0.0054
  Layer  21: 0.0051
  Layer  22: 0.0048
  Layer  23: 0.0047
  Layer  24: 0.0045
  Layer  25: 0.0042
  Layer  26: 0.0039
  Layer  27: 0.0035
  Layer  28: 0.0032
  Layer  29: 0.0031
  Layer  30: 0.0032

--------------------------------------------------------------------------------
user
How was your day?
model
As an AI, I don't really *have* days in the same way humans do! But my systems have been running smoothly and I’ve been busy assisting users with all sorts of requests – answering questions, writing stories, translating languages, and even just chatting. 

It’s been a productive and i

## 7. Comparison Summary

In [11]:
print("="*80)
print("SUMMARY OF FINDINGS")
print("="*80)

print("\n1. NORMALIZED VECTOR:")
print(f"   - Vector norms: ~1.0 (unit normalized)")
print(f"   - Working coefficient range: 0.5 - 5.0")
print(f"   - Behavior: Consistent, predictable steering")

print("\n2. UNNORMALIZED VECTOR:")
first_layer = list(unnormalized_vector.directions.keys())[0]
unnorm_norm = np.linalg.norm(unnormalized_vector.directions[first_layer])
print(f"   - Vector norms: ~{unnorm_norm:.6f} (not normalized)")
print(f"   - Required coefficient range: {int(1/unnorm_norm * 2)}-{int(1/unnorm_norm * 5)}")
print(f"   - Behavior: Requires MUCH larger coefficients (300-500)")

if normalized_vector.activation_norms:
    print("\n3. ADAPTIVE SCALING:")
    print(f"   - Automatically adjusts per layer based on activation magnitude")
    print(f"   - Base coefficient: 0.5 - 5.0")
    print(f"   - Behavior: More nuanced, layer-specific steering")

print("\n" + "="*80)
print("\n✓ CONCLUSION:")
print("  The normalization issue was causing the need for 300-500 coefficients.")
print("  With proper normalization, coefficients in the 0.5-5.0 range work well!")
print("="*80)

SUMMARY OF FINDINGS

1. NORMALIZED VECTOR:
   - Vector norms: ~1.0 (unit normalized)
   - Working coefficient range: 0.5 - 5.0
   - Behavior: Consistent, predictable steering

2. UNNORMALIZED VECTOR:
   - Vector norms: ~1.000000 (not normalized)
   - Required coefficient range: 2-5
   - Behavior: Requires MUCH larger coefficients (300-500)

3. ADAPTIVE SCALING:
   - Automatically adjusts per layer based on activation magnitude
   - Base coefficient: 0.5 - 5.0
   - Behavior: More nuanced, layer-specific steering


✓ CONCLUSION:
  The normalization issue was causing the need for 300-500 coefficients.
  With proper normalization, coefficients in the 0.5-5.0 range work well!


## 8. Export Normalized Vector

In [ ]:
# Export the properly normalized vector
output_path = "sentiment_happy_sad_normalized.gguf"
normalized_vector.export_gguf(output_path)
print(f"Exported normalized vector to: {output_path}")

# Also save analysis
import json
from gemma3_adapter import analyze_vector_properties

analysis = analyze_vector_properties(normalized_vector)
# Convert numpy types to native Python for JSON serialization
analysis_json = {
    "num_layers": analysis["num_layers"],
    "layer_ids": analysis["layer_ids"],
    "vector_norms": {str(k): float(v) for k, v in analysis["vector_norms"].items()},
    "activation_norms": {str(k): float(v) for k, v in analysis["activation_norms"].items()}
}

with open("sentiment_vector_analysis.json", "w") as f:
    json.dump(analysis_json, f, indent=2)
print("Saved analysis to: sentiment_vector_analysis.json")

## 9. Next Steps

Now that we've verified the adapter works with reasonable coefficients:

1. **Use this adapter for ToM training** - Replace the training code in your ToM notebook
2. **Test with your ToM dataset** - Train the specialized ToM vectors with normalization
3. **Verify coefficients** - Ensure ToM steering works with coefficients ~1-5 instead of 300-500
4. **Try adaptive scaling** - Test if per-layer scaling improves ToM performance

The key insight: **Vector normalization is essential for stable, predictable steering!**